In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import importlib


In [3]:
import copy
import sys
import os

sys.path.append(os.path.abspath("../"))
from campaign_diagram import *

## IN this notebook
# guard_position, dual mode, etc....

Let us specify 4 kernels, but this time, some kernels overlap in time.

In [4]:
kernels1= [
    Kernel("K01", 0, 7, .50, .40),  # Kernel from time 0 to 7
    Kernel("K02", 2, 2, .40, .10),  # Kernel from time 2 to 4
    Kernel("K03", 3, 6, .60, .30),  # Kernel from time 2 to 8
    Kernel("K04", 8, 4, .40, .90),  # Kernel from time 1 to 5
]

cascade1 = Cascade(name="Simple Cascade",
                   kernels=kernels1)


In [5]:
print("")
cascade1.pretty_print()
print("")


Cascade: Simple Cascade
Kernel(name=K01, start=0.00, duration=2.00, throttled_duration=1.00, compute_util=0.50, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K02, start=2.00, duration=1.00, throttled_duration=0.60, compute_util=0.40, bw_util=0.10,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K01, start=2.00, duration=1.00, throttled_duration=0.50, compute_util=0.50, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K03, start=3.00, duration=1.00, throttled_duration=0.40, compute_util=0.60, bw_util=0.30,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K02, start=3.00, duration=1.00, throttled_duration=0.60, compute_util=0.40, bw_util=0.10,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K01, start=3.00, duration=1.00, throttled_duration=0.50, compute_util=0.50, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K03, star

#### Now, display the cascade.
Some observations:
- Timesteps 0 to 2:
  - Einsum K01 runs with a compute resource utiliation of 0.5 and memory bandwidth utilization of 0.4.
- Timesteps 2 to 3:
  - Einsum K02 joins, running at the same time as Einsum K01.
  - Einsum K02 has a compute utilization of 0.4 and a memory bandwidth utilization of 0.1.
  - Note that with Einsum K01 already using 0.5 of the compute resources, adding Einsum K02 means we are now using a total of **0.9** of the compute resources (K02 compute line is at 0.9).
  - Note that with Einsum K01 already using 0.4 of the memory bandwidth, there is only 0.6 of the memory bandwidth left for any other Einsum. Thus, the memory well/pipe (height of the guard lines) is smaller for the K02 Einsum block in the figure.
  - Compute resource utilization: <mark> overflow! </mark>
- Timesteps 3-4:
  - Einsum K03 joins in, with compute utilization of 0.6 and memory bandwidth utilization of 0.3.
  - Compute resource utilization: <mark> overflow! </mark>

- Timesteps 4-7:
  - Einsum K02 finishes at timestep 4.
  - Compute resource utilization: <mark> overflow! </mark>
    
- Timestep 7-8:
  - Einsum K01 finishes at timestep 7.
  - Only Einsum K02 is running now.
    
- Timesteps 8-9:
  - Einsum K04 joins, with compute resource utilization of 0.4, and memory bandwidth utilization of 0.9.
  - memory bandwidth utilization: <mark> overflow! </mark>

<mark> TODO: axis as "cumulative compute utilization" </mark>

In [6]:
campaign_diagram = CampaignDiagram(cascade1)
campaign_diagram.draw().interactive().show()

campaign_diagram.draw(dual_mode=True).interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

#### Notice that at some points, different resources overlow.
- To address this, we *throttle.*
- When a particular resource is overbooked at a timestamp, throttling reduces the utilizations of all Einsums running at that time so that said resource is at 100% utilization.
- Below, we display the throttled cascade (call `cascade.throttle()`)
    - Now, at all timestamps, no resource (memory bandwidth nor compute) are overutilized.
    - Dashed lines represent the **extra time** required to run that portion of the Einsum because of throttling.

- <mark> Note: we could have specified a priority order of kernels to throttle, instead of throttling uniformly </mark>
- <mark> TODO: make throttle a default (so add a boolean key) </mark>

In [7]:
#throttle
throttled_cascade = cascade1.throttle()
campaign_diagram = CampaignDiagram(throttled_cascade)
campaign_diagram.draw().interactive().show()

campaign_diagram.draw(dual_mode=True).interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

<mark> TODO: all of the notebooks -- need to add paragraphs to explain what is going on. Ask JDO to review :) </mark>

In [8]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import importlib


In [10]:
import copy
import sys
import os

sys.path.append(os.path.abspath("../"))
from campaign_diagram import *

## IN this notebook
# guard_position, dual mode, etc....

Let us specify 4 kernels, but this time, some kernels overlap in time.

In [11]:
kernels1= [
    Kernel("K01", 0, 7, .50, .40),  # Kernel from time 0 to 7
    Kernel("K02", 2, 2, .40, .10),  # Kernel from time 2 to 4
    Kernel("K03", 3, 6, .60, .30),  # Kernel from time 2 to 8
    Kernel("K04", 8, 4, .40, .90),  # Kernel from time 1 to 5
]

cascade1 = Cascade(name="Simple Cascade",
                   kernels=kernels1)


In [12]:
print("")
cascade1.pretty_print()
print("")


Cascade: Simple Cascade
Kernel(name=K01, start=0.00, duration=2.00, throttled_duration=1.00, compute_util=0.50, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K02, start=2.00, duration=1.00, throttled_duration=0.60, compute_util=0.40, bw_util=0.10,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K01, start=2.00, duration=1.00, throttled_duration=0.50, compute_util=0.50, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K03, start=3.00, duration=1.00, throttled_duration=0.40, compute_util=0.60, bw_util=0.30,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K02, start=3.00, duration=1.00, throttled_duration=0.60, compute_util=0.40, bw_util=0.10,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K01, start=3.00, duration=1.00, throttled_duration=0.50, compute_util=0.50, bw_util=0.40,fusion_group=None,fusion_type=FusionType.NONE,windup=0.0

Kernel(name=K03, star

#### Now, display the cascade.
Some observations:
- Timesteps 0 to 2:
  - Einsum K01 runs with a compute resource utiliation of 0.5 and memory bandwidth utilization of 0.4.
- Timesteps 2 to 3:
  - Einsum K02 joins, running at the same time as Einsum K01.
  - Einsum K02 has a compute utilization of 0.4 and a memory bandwidth utilization of 0.1.
  - Note that with Einsum K01 already using 0.5 of the compute resources, adding Einsum K02 means we are now using a total of **0.9** of the compute resources (K02 compute line is at 0.9).
  - Note that with Einsum K01 already using 0.4 of the memory bandwidth, there is only 0.6 of the memory bandwidth left for any other Einsum. Thus, the memory well/pipe (height of the guard lines) is smaller for the K02 Einsum block in the figure.
  - Compute resource utilization: <mark> overflow! </mark>
- Timesteps 3-4:
  - Einsum K03 joins in, with compute utilization of 0.6 and memory bandwidth utilization of 0.3.
  - Compute resource utilization: <mark> overflow! </mark>

- Timesteps 4-7:
  - Einsum K02 finishes at timestep 4.
  - Compute resource utilization: <mark> overflow! </mark>
    
- Timestep 7-8:
  - Einsum K01 finishes at timestep 7.
  - Only Einsum K02 is running now.
    
- Timesteps 8-9:
  - Einsum K04 joins, with compute resource utilization of 0.4, and memory bandwidth utilization of 0.9.
  - memory bandwidth utilization: <mark> overflow! </mark>

<mark> TODO: axis as "cumulative compute utilization" </mark>

In [13]:
campaign_diagram = CampaignDiagram(cascade1)
campaign_diagram.draw().interactive().show()

campaign_diagram.draw(dual_mode=True).interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

#### Notice that at some points, different resources overlow.
- To address this, we *throttle.*
- When a particular resource is overbooked at a timestamp, throttling reduces the utilizations of all Einsums running at that time so that said resource is at 100% utilization.
- Below, we display the throttled cascade (call `cascade.throttle()`)
    - Now, at all timestamps, no resource (memory bandwidth nor compute) are overutilized.
    - Dashed lines represent the **extra time** required to run that portion of the Einsum because of throttling.

- <mark> Note: we could have specified a priority order of kernels to throttle, instead of throttling uniformly </mark>
- <mark> TODO: make throttle a default (so add a boolean key) </mark>

In [14]:
#throttle
throttled_cascade = cascade1.throttle()
campaign_diagram = CampaignDiagram(throttled_cascade)
campaign_diagram.draw().interactive().show()

campaign_diagram.draw(dual_mode=True).interactive().show()

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

~/.local/lib/python3.10/site-packages/altair/vegalite/v6/api.py:3804: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

<mark> TODO: all of the notebooks -- need to add paragraphs to explain what is going on. Ask JDO to review :) </mark>